# 🔁 03 · 留存分析与同期群（Cohort）

> 留存是增长分析的灵魂。本节讲清楚**留存率口径、同期群矩阵、留存曲线**，并手写 vs pandas 对照。

## 📋 本节要解决的问题

| # | 问题 | 工具 |
|---|---|---|
| 1 | 留存率为什么按「同期群」看？ | 注册日分组 |
| 2 | 同期群矩阵怎么算？ | 活跃日 - 注册日 = 第 N 天 |
| 3 | 留存曲线怎么画？ | 逐期留存率 |
| 4 | 留存差意味着什么？ | 版本/渠道对比 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
np.random.seed(42)


## 1️⃣ 构造注册-活跃数据

每个用户有注册日，以及若干活跃日：

In [2]:
rng = np.random.default_rng(11)
reg_days = rng.integers(1, 15, 600)              # 注册日（第 1~14 天）
act_days = rng.integers(1, 22, 600)              # 活跃日
active = pd.DataFrame({'reg_day': reg_days, 'act_day': act_days})
active = active[active['act_day'] >= active['reg_day']]  # 活跃日不早于注册日
print('记录数:', len(active))
print(active.head())

记录数: 414
   reg_day  act_day
0        2       12
3        7       17
4        9       19
7        1        1
8        7       17


## 2️⃣ 同期群矩阵：第 N 天留存率

In [3]:
# 每个 (注册日, 活跃日) 对 → 留存天数 = 活跃日 - 注册日
a = active.copy()
a['retention_day'] = a['act_day'] - a['reg_day']
# 同期群（注册日）× 留存天 的 去重用户数
cohort_size = a.groupby('reg_day')['act_day'].nunique().rename('base')
cohort_ret = a.groupby(['reg_day', 'retention_day'])['act_day'].nunique().unstack(fill_value=0)

# 留存率矩阵 = 各期人数 / 注册日基础人数
ret_matrix = cohort_ret.div(cohort_size, axis=0)
print('同期群留存率矩阵（前 6 行，第 0 期恒为 1）:')
print(ret_matrix.iloc[:6, :6].round(3))

同期群留存率矩阵（前 6 行，第 0 期恒为 1）:
retention_day      0      1      2      3      4      5
reg_day                                                
1              0.050  0.000  0.050  0.050  0.050  0.050
2              0.050  0.050  0.050  0.050  0.050  0.050
3              0.059  0.059  0.059  0.059  0.059  0.000
4              0.077  0.077  0.077  0.077  0.077  0.000
5              0.059  0.059  0.059  0.059  0.059  0.059
6              0.071  0.071  0.071  0.071  0.071  0.071


## 3️⃣ 留存曲线：整体逐期留存

In [4]:
overall = a.groupby('retention_day')['act_day'].nunique() / cohort_size.sum()
overall = overall.sort_index()

plt.figure(figsize=(7, 4))
plt.plot(overall.index, overall.values, marker='o', color='#2563eb')
plt.xlabel('注册后第 N 天')
plt.ylabel('留存率')
plt.title('整体留存曲线（模拟）')
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.show()
# retention_day=0 的活跃用户数 ≤ 注册数（同一天注册且活跃可去重），留存率必 ≤1
assert overall.iloc[0] <= 1.0
print('✅ 留存曲线绘制完成（第 0 期留存 ≤1 由定义保证）')

✅ 留存曲线绘制完成（第 0 期留存 ≤1 由定义保证）


C:\Users\24260\AppData\Local\Temp\ipykernel_2640\2773887632.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4️⃣ 同期群对比：A/B 两个版本留存

模拟两个版本的次日留存：

In [5]:
rng2 = np.random.default_rng(5)
vA = rng2.normal(0.45, 0.05, 30)
vB = rng2.normal(0.50, 0.05, 30)

plt.figure(figsize=(7, 4))
plt.plot(vA, label='版本 A', color='#f59e0b')
plt.plot(vB, label='版本 B', color='#10b981')
plt.xlabel('实验批次')
plt.ylabel('次日留存率')
plt.title('A/B 版本次日留存对比')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(f'A 平均留存 {vA.mean():.1%}，B 平均留存 {vB.mean():.1%}')
print(f'B 相对提升 {(vB.mean()/vA.mean()-1):.1%}')
assert vB.mean() > vA.mean()
print('✅ B 版本留存更高（模拟设定）')

A 平均留存 43.9%，B 平均留存 48.9%
B 相对提升 11.4%
✅ B 版本留存更高（模拟设定）


C:\Users\24260\AppData\Local\Temp\ipykernel_2640\365935425.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5️⃣ 留存分析常见误区

In [6]:
# 误区：把不同注册日的用户混在一起算留存 → 掩盖真实趋势
# 演示：混合计算 vs 同期群计算
mixed = a.groupby('retention_day')['act_day'].nunique() / a.groupby('retention_day')['act_day'].count()
print('混合口径（分母用活跃记录数，错误）:')
print(mixed.iloc[:3].round(3))
print('\n同期群口径（分母用注册日基础用户数，正确）: 见上方矩阵')
print('→ 口径不同结论不同，面试常考「分母是什么」')

混合口径（分母用活跃记录数，错误）:
retention_day
0    0.333
1    0.526
2    0.433
Name: act_day, dtype: float64

同期群口径（分母用注册日基础用户数，正确）: 见上方矩阵
→ 口径不同结论不同，面试常考「分母是什么」


## 📝 自测清单

- [ ] 能解释「同期群」解决什么问题（用户分层随时间变化）
- [ ] 能手写留存矩阵：注册日 × 留存天数
- [ ] 能画出留存曲线并解读衰减形态
- [ ] 能说出留存率的分母/分子口径并识别错误口径